# Dijkstra con cubetas de Dial — demostración

Tarea individual: hallar la distancia entre dos vértices leyendo el grafo de un archivo de texto.
Límites del enunciado: `1 ≤ V ≤ 1.000.000`, `1 ≤ E ≤ 500.000`, pesos enteros `1 ≤ w ≤ 100`.

**Implementación:** C, algoritmo de **Dial** en lugar de un heap. Como los pesos son enteros de 1 a 100,
todas las distancias que hay en la cola en un instante caben en la ventana `[d, d+100]`, así que
**101 cubetas circulares** bastan y sacar el mínimo cuesta *O(1)* en vez de *O(log n)*, sin comparaciones.

Otras decisiones:

- **máscara de 128 bits + `__builtin_ctzll`** para saltar de golpe a la siguiente cubeta no vacía;
- **borrado perezoso** en vez de `decrease-key`: se insertan duplicados y se descartan con `dist[u] != dist_actual`;
- **terminación temprana** al extraer el destino;
- grafo en **arreglos planos** (`head`/`next`, `tw = (destino<<7)|peso`), construido en la misma pasada del parseo;
- entrada con **`mmap`** y conversión de dígitos a mano (sin `scanf`), porque con 1.000.000 de etiquetas
  el archivo pesa ~15 MB y **leerlo cuesta más que ejecutar Dijkstra**.

Este notebook es autocontenido: las celdas de la sección 1 escriben todo el código fuente en el
disco de Colab, así que no hace falta descargar ni clonar nada. Ejecutar con
*Entorno de ejecución → Ejecutar todas*.

## 1. El código fuente

Siete archivos. Los tres primeros son el programa que se entrega; los demás son el banco de
pruebas y la verificación.

| archivo | qué es |
|---|---|
| `graph.h` | lectura con `mmap` y parseo de la entrada |
| `queues.h` | las cuatro colas de prioridad sobre el mismo Dijkstra |
| `dijkstra.c` | el programa final |
| `bench.c` | cronometra el parseo y cada cola sobre un mismo grafo |
| `gen.py` | genera los grafos de prueba (cadena, malla, disperso, denso) |
| `chart.py` | convierte los resultados en un gráfico |
| `test.sh`, `bench.sh` | pruebas de correctitud y ejecución del benchmark |

In [ ]:
%%writefile graph.h
/* Carga y parseo de la entrada (compartido por dijkstra.c y bench.c). */
#ifndef GRAPH_H
#define GRAPH_H
#define _GNU_SOURCE
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <fcntl.h>
#include <sys/stat.h>
#include <sys/mman.h>

#define UNDIRECTED 1              /* "arista entre u y v" => doble sentido */
#define WMAX 100
#define INF 0xffffffffu

struct Node { unsigned dist; int head; };   /* dist y head juntos: una linea de cache */
struct Arc  { int tw; int nx; };            /* tw = (destino << 7) | peso */

struct Graph {
    unsigned V, E, s, t;
    size_t narcs;
    struct Node *node;   /* indices 1..V */
    struct Arc *arc;
};

static void die(const char *m) { fprintf(stderr, "error: %s\n", m); exit(1); }

static inline unsigned rd(const char **pp, const char *end) {
    const char *p = *pp;
    while (p < end && (unsigned)(*p - '0') > 9u) p++;
    unsigned x = 0, c;
    while (p < end && (c = (unsigned)(*p - '0')) <= 9u) { x = x * 10 + c; p++; }
    *pp = p;
    return x;
}

/* Salta n lineas contando '\n' en bloques de 64 (el compilador lo vectoriza):
   un memchr por linea costaba 1M de llamadas = ~5 ms con V=1.000.000. */
static const char *skip_lines(const char *p, const char *end, long n) {
    while (n > 0 && end - p >= 64) {
        long c = 0;
        for (int i = 0; i < 64; i++) c += (p[i] == '\n');
        if (c >= n) break;   /* el salto n-esimo esta en este bloque: termina byte a byte */
        n -= c; p += 64;
    }
    while (n > 0 && p < end) { if (*p == '\n') n--; p++; }
    return p;
}

/* Paginas grandes (2 MB): menos fallos de pagina al inicializar los 8 MB de
   nodos y menos fallos de TLB en los accesos aleatorios durante el parseo. */
static void *gr_alloc(size_t n) {
    void *m = mmap(NULL, n, PROT_READ | PROT_WRITE, MAP_PRIVATE | MAP_ANONYMOUS, -1, 0);
    if (m == MAP_FAILED) die("sin memoria");
    madvise(m, n, MADV_HUGEPAGE);
    return m;
}

/* libera lo que reservo gr_parse (solo lo usa bench.c al repetir el parseo) */
static void gr_free(struct Graph *g) {
    if (g->node) munmap(g->node, (size_t)(g->V + 1) * sizeof *g->node);
    if (g->arc) munmap(g->arc, (g->narcs ? g->narcs : 1) * sizeof *g->arc);
    g->node = NULL; g->arc = NULL;
}

static char *slurp_fd(int fd, size_t *len) {
    size_t cap = 1u << 22, n = 0;
    char *b = malloc(cap);
    if (!b) return NULL;
    for (;;) {
        if (n == cap) { cap *= 2; char *tb = realloc(b, cap); if (!tb) { free(b); return NULL; } b = tb; }
        ssize_t r = read(fd, b + n, cap - n);
        if (r <= 0) break;
        n += (size_t)r;
    }
    *len = n;
    return b;
}

/* mmap del archivo (o lectura completa si es stdin/pipe) */
static void gr_load(const char *path, const char **pp, const char **pend) {
    if (!path) {
        size_t n; char *b = slurp_fd(0, &n);
        if (!b) die("sin memoria");
        *pp = b; *pend = b + n;
        return;
    }
    int fd = open(path, O_RDONLY);
    if (fd < 0) die("no se pudo abrir el archivo");
    struct stat st;
    char *m = MAP_FAILED;
    size_t sz = 0;
    if (!fstat(fd, &st) && S_ISREG(st.st_mode)) {
        sz = (size_t)st.st_size;
        m = sz ? mmap(NULL, sz, PROT_READ, MAP_PRIVATE | MAP_POPULATE, fd, 0) : (char *)"";
    }
    if (m == MAP_FAILED) {
        size_t n; char *b = slurp_fd(fd, &n);
        if (!b) die("sin memoria");
        *pp = b; *pend = b + n;
    } else {
        *pp = m; *pend = m + sz;
    }
    close(fd);
}

/* parsea cabecera + etiquetas + aristas y construye las listas de adyacencia */
static void gr_parse(const char *p, const char *end, struct Graph *g) {
    g->V = rd(&p, end); g->E = rd(&p, end);
    g->s = rd(&p, end); g->t = rd(&p, end);
    unsigned V = g->V, E = g->E;
    if (V < 1 || g->s < 1 || g->s > V || g->t < 1 || g->t > V) die("V, s o t fuera de rango");
    if (g->s == g->t) { g->node = NULL; g->arc = NULL; g->narcs = 0; return; }   /* nada que parsear */

    p = skip_lines(p, end, (long)V + 1);   /* resto de la linea 2 + V etiquetas */

    g->narcs = (size_t)E * (UNDIRECTED ? 2 : 1);
    g->node = gr_alloc((size_t)(V + 1) * sizeof *g->node);
    g->arc = gr_alloc((g->narcs ? g->narcs : 1) * sizeof *g->arc);
    if (!g->node || !g->arc) die("sin memoria");
    memset(g->node, 0xff, (size_t)(V + 1) * sizeof *g->node);   /* dist=INF, head=-1 */

    struct Node *node = g->node;
    struct Arc *arc = g->arc;
    int na = 0;
    for (unsigned i = 0; i < E; i++) {
        unsigned u = rd(&p, end), v = rd(&p, end), w = rd(&p, end);
        if (u < 1 || u > V || v < 1 || v > V || w < 1 || w > WMAX) die("arista fuera de rango");
        arc[na].tw = (int)((v << 7) | w); arc[na].nx = node[u].head; node[u].head = na++;
#if UNDIRECTED
        arc[na].tw = (int)((u << 7) | w); arc[na].nx = node[v].head; node[v].head = na++;
#endif
    }
}
#endif

In [ ]:
%%writefile queues.h
/* Cuatro colas de prioridad para el mismo Dijkstra, para compararlas.
 * Cada solver recibe el grafo (node[] mutable: usa node[].dist como scratch)
 * y devuelve la distancia s->t, o INF si no hay camino. */
#ifndef QUEUES_H
#define QUEUES_H
#include <stdint.h>
#include "graph.h"

#define NB (WMAX + 1)   /* 101 cubetas circulares de Dial */

struct Ent { int node; int next; };   /* entrada de cubeta (lista enlazada en pool) */

static inline int ctz128(unsigned __int128 x) {
    unsigned long long lo = (unsigned long long)x;
    return lo ? __builtin_ctzll(lo) : 64 + __builtin_ctzll((unsigned long long)(x >> 64));
}

/* ---- 1) Dial con mascara de bits (salta cubetas vacias en O(1)) ---- */
static unsigned solve_dial(struct Node *node, const struct Arc *arc,
                           unsigned s, unsigned t, struct Ent *pool) {
    int bhead[NB];
    memset(bhead, 0xff, sizeof bhead);
    unsigned __int128 mask = 1;
    int np = 0, b = 0;
    unsigned curd = 0;

    node[s].dist = 0;
    pool[np].node = (int)s; pool[np].next = -1; bhead[0] = np++;

    while (mask) {
        unsigned __int128 hi = mask >> b;                  /* cubetas en [b, 100] */
        int k = hi ? ctz128(hi) : NB - b + ctz128(mask);   /* si no, da la vuelta */
        curd += (unsigned)k;
        b += k; if (b >= NB) b -= NB;

        for (int e = bhead[b]; e >= 0; e = bhead[b]) {
            bhead[b] = pool[e].next;
            unsigned u = (unsigned)pool[e].node;
            if (node[u].dist != curd) continue;            /* entrada obsoleta */
            if (u == t) return curd;
            for (int a = node[u].head; a >= 0; a = arc[a].nx) {
                unsigned tw = (unsigned)arc[a].tw, v = tw >> 7, w = tw & 127;
                unsigned nd = curd + w;
                if (nd < node[v].dist) {
                    node[v].dist = nd;
                    int b2 = b + (int)w; if (b2 >= NB) b2 -= NB;   /* w<=100 => b2!=b */
                    pool[np].node = (int)v; pool[np].next = bhead[b2]; bhead[b2] = np++;
                    mask |= (unsigned __int128)1 << b2;
                }
            }
        }
        mask &= ~((unsigned __int128)1 << b);
    }
    return INF;
}

/* ---- 2) Dial sin mascara: busca la siguiente cubeta recorriendolas ---- */
static unsigned solve_dial_scan(struct Node *node, const struct Arc *arc,
                                unsigned s, unsigned t, struct Ent *pool) {
    int bhead[NB];
    memset(bhead, 0xff, sizeof bhead);
    int np = 0, b = 0, pend = 1;
    unsigned curd = 0;

    node[s].dist = 0;
    pool[np].node = (int)s; pool[np].next = -1; bhead[0] = np++;

    while (pend > 0) {
        while (bhead[b] < 0) { b++; if (b >= NB) b -= NB; curd++; }
        for (int e = bhead[b]; e >= 0; e = bhead[b]) {
            bhead[b] = pool[e].next;
            pend--;
            unsigned u = (unsigned)pool[e].node;
            if (node[u].dist != curd) continue;
            if (u == t) return curd;
            for (int a = node[u].head; a >= 0; a = arc[a].nx) {
                unsigned tw = (unsigned)arc[a].tw, v = tw >> 7, w = tw & 127;
                unsigned nd = curd + w;
                if (nd < node[v].dist) {
                    node[v].dist = nd;
                    int b2 = b + (int)w; if (b2 >= NB) b2 -= NB;
                    pool[np].node = (int)v; pool[np].next = bhead[b2]; bhead[b2] = np++;
                    pend++;
                }
            }
        }
    }
    return INF;
}

/* clave empaquetada: distancia en los bits altos, nodo en los 20 bajos
   => comparar prioridades es comparar un solo uint64 */
#define KEY(d, v) (((uint64_t)(d) << 20) | (uint64_t)(v))

/* ---- 3) Heap binario, borrado perezoso ---- */
static unsigned solve_heap_bin(struct Node *node, const struct Arc *arc,
                               unsigned s, unsigned t, uint64_t *h) {
    size_t n = 0;
    node[s].dist = 0;
    h[n++] = KEY(0, s);
    while (n) {
        uint64_t top = h[0];
        uint64_t last = h[--n];
        if (n) {                                  /* baja el ultimo desde la raiz */
            size_t i = 0;
            for (;;) {
                size_t c = 2 * i + 1;
                if (c >= n) break;
                if (c + 1 < n && h[c + 1] < h[c]) c++;
                if (h[c] >= last) break;
                h[i] = h[c]; i = c;
            }
            h[i] = last;
        }
        unsigned u = (unsigned)(top & 0xfffff), curd = (unsigned)(top >> 20);
        if (node[u].dist != curd) continue;
        if (u == t) return curd;
        for (int a = node[u].head; a >= 0; a = arc[a].nx) {
            unsigned tw = (unsigned)arc[a].tw, v = tw >> 7, w = tw & 127;
            unsigned nd = curd + w;
            if (nd < node[v].dist) {
                node[v].dist = nd;
                uint64_t k = KEY(nd, v);
                size_t i = n++;
                while (i && h[(i - 1) / 2] > k) { h[i] = h[(i - 1) / 2]; i = (i - 1) / 2; }
                h[i] = k;
            }
        }
    }
    return INF;
}

/* ---- 4) Heap 4-ario (mas plano = menos fallos de cache) ---- */
static unsigned solve_heap_4ary(struct Node *node, const struct Arc *arc,
                                unsigned s, unsigned t, uint64_t *h) {
    size_t n = 0;
    node[s].dist = 0;
    h[n++] = KEY(0, s);
    while (n) {
        uint64_t top = h[0];
        uint64_t last = h[--n];
        if (n) {
            size_t i = 0;
            for (;;) {
                size_t c = 4 * i + 1;
                if (c >= n) break;
                size_t lim = c + 4 < n ? c + 4 : n, best = c;
                for (size_t j = c + 1; j < lim; j++) if (h[j] < h[best]) best = j;
                if (h[best] >= last) break;
                h[i] = h[best]; i = best;
            }
            h[i] = last;
        }
        unsigned u = (unsigned)(top & 0xfffff), curd = (unsigned)(top >> 20);
        if (node[u].dist != curd) continue;
        if (u == t) return curd;
        for (int a = node[u].head; a >= 0; a = arc[a].nx) {
            unsigned tw = (unsigned)arc[a].tw, v = tw >> 7, w = tw & 127;
            unsigned nd = curd + w;
            if (nd < node[v].dist) {
                node[v].dist = nd;
                uint64_t k = KEY(nd, v);
                size_t i = n++;
                while (i && h[(i - 1) / 4] > k) { h[i] = h[(i - 1) / 4]; i = (i - 1) / 4; }
                h[i] = k;
            }
        }
    }
    return INF;
}
#endif

In [ ]:
%%writefile dijkstra.c
/* Dijkstra con pesos enteros 1..100.
 * Uso: ./dijkstra entrada.txt [cola]   |   ./dijkstra - [cola] < entrada.txt
 *      cola: dial (por defecto) | barrido | heap | heap4   -- ver queues.h
 *      TIEMPO=1 ./dijkstra ...  escribe el desglose de tiempos en stderr
 * Formato: "V E" / "s t" / V lineas de etiquetas / E lineas "u v w". */
#include <time.h>
#include "queues.h"

#define NO_PATH "No hay camino"   /* cambiar por "-1" si lo piden asi */
#define USO "cola desconocida; usa: dial | barrido | heap | heap4"

static double ms(void) {   /* CLOCK_MONOTONIC via vDSO: ~25 ns por llamada */
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);
    return t.tv_sec * 1e3 + t.tv_nsec / 1e6;
}

int main(int argc, char **argv) {
    double t0 = ms();
    const char *path = (argc > 1 && strcmp(argv[1], "-")) ? argv[1] : NULL;
    const char *cola = argc > 2 ? argv[2] : "dial";

    const char *p, *end;
    gr_load(path, &p, &end);

    struct Graph g;
    gr_parse(p, end, &g);
    double t1 = ms();

    unsigned d = 0;   /* s == t => 0, y gr_parse no llego a construir el grafo */
    if (g.s != g.t) {
        /* una sola reserva: las cubetas de Dial y el heap usan 8 bytes por entrada */
        void *scratch = malloc((g.narcs + 2) * 8);
        if (!scratch) die("sin memoria");
        d = !strcmp(cola, "dial")    ? solve_dial(g.node, g.arc, g.s, g.t, scratch)
          : !strcmp(cola, "barrido") ? solve_dial_scan(g.node, g.arc, g.s, g.t, scratch)
          : !strcmp(cola, "heap")    ? solve_heap_bin(g.node, g.arc, g.s, g.t, scratch)
          : !strcmp(cola, "heap4")   ? solve_heap_4ary(g.node, g.arc, g.s, g.t, scratch)
          : (die(USO), 0u);
    }
    double t2 = ms();

    if (d == INF) puts(NO_PATH);
    else printf("%u\n", d);

    /* a stderr: la salida del programa sigue siendo solo la distancia */
    if (getenv("TIEMPO"))
        fprintf(stderr, "parseo %.3f ms | %s %.3f ms | total %.3f ms\n",
                t1 - t0, cola, t2 - t1, t2 - t0);
    return 0;
}

In [ ]:
%%writefile bench.c
/* Mide parseo y cada cola de prioridad sobre el mismo grafo.
 * Uso: ./bench entrada.txt [repeticiones]   -> una linea CSV por variante */
#include <time.h>
#include "queues.h"

static double ms_now(void) {
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec * 1e3 + ts.tv_nsec / 1e6;
}

static int cmpd(const void *a, const void *b) {
    double x = *(const double *)a, y = *(const double *)b;
    return (x > y) - (x < y);
}

int main(int argc, char **argv) {
    if (argc < 2) die("uso: ./bench entrada.txt [repeticiones]");
    int R = argc > 2 ? atoi(argv[2]) : 5;
    if (R < 1 || R > 99) R = 5;
    const char *name = strrchr(argv[1], '/');
    name = name ? name + 1 : argv[1];

    double *tm = malloc((size_t)R * sizeof *tm);
    struct Graph g;
    const char *p, *end;

    /* parseo: se repite releyendo el archivo (incluye E/S + construccion) */
    for (int r = 0; r < R; r++) {
        double t0 = ms_now();
        gr_load(argv[1], &p, &end);
        gr_parse(p, end, &g);
        tm[r] = ms_now() - t0;
        if (r + 1 < R) gr_free(&g);
    }
    qsort(tm, (size_t)R, sizeof *tm, cmpd);
    printf("%s,%u,%u,parseo,%.3f,%.3f,-\n", name, g.V, g.E, tm[0], tm[R / 2]);
    if (!g.node) die("s == t: nada que medir");

    size_t nb = (size_t)(g.V + 1) * sizeof(struct Node);
    struct Node *pristine = malloc(nb), *work = malloc(nb);
    struct Ent *pool = malloc((g.narcs + 2) * sizeof *pool);
    uint64_t *heap = malloc((g.narcs + 2) * sizeof *heap);
    if (!pristine || !work || !pool || !heap) die("sin memoria");
    memcpy(pristine, g.node, nb);

    const char *names[] = { "Dial+mascara", "Dial+barrido", "Heap binario", "Heap 4-ario" };
    unsigned res[4];
    for (int v = 0; v < 4; v++) {
        for (int r = 0; r < R; r++) {
            memcpy(work, pristine, nb);          /* fuera del cronometro */
            double t0 = ms_now();
            unsigned d = v == 0 ? solve_dial(work, g.arc, g.s, g.t, pool)
                       : v == 1 ? solve_dial_scan(work, g.arc, g.s, g.t, pool)
                       : v == 2 ? solve_heap_bin(work, g.arc, g.s, g.t, heap)
                                : solve_heap_4ary(work, g.arc, g.s, g.t, heap);
            tm[r] = ms_now() - t0;
            res[v] = d;
        }
        qsort(tm, (size_t)R, sizeof *tm, cmpd);
        printf("%s,%u,%u,%s,%.3f,%.3f,", name, g.V, g.E, names[v], tm[0], tm[R / 2]);
        if (res[v] == INF) puts("sin camino"); else printf("%u\n", res[v]);
    }
    for (int v = 1; v < 4; v++)
        if (res[v] != res[0]) { fprintf(stderr, "DISCREPANCIA en %s: %s=%u vs Dial=%u\n", name, names[v], res[v], res[0]); return 1; }
    return 0;
}

In [ ]:
%%writefile gen.py
"""Genera grafos de prueba en t/ con distintas formas (peso 1..100)."""
import random, os
random.seed(7)
os.makedirs("t", exist_ok=True)

def write(name, V, edges, s, t):
    with open(f"t/{name}.txt", "w") as f:
        f.write(f"{V} {len(edges)}\n{s} {t}\n")
        f.write("".join(f"n{i}\n" for i in range(1, V + 1)))
        f.write("".join(f"{u} {v} {w}\n" for u, v, w in edges))
    print(f"t/{name}.txt  V={V} E={len(edges)}")

w = lambda: random.randint(1, 100)

# cadena larga: distancias enormes (~2.5e7), peor caso para saltar cubetas
write("cadena-1M", 1_000_000, [(i, i + 1, w()) for i in range(1, 500_001)], 1, 500_001)

# malla 500x500: parecido a una red de calles
n = 500
E = [(r * n + c + 1, r * n + c + 2, w()) for r in range(n) for c in range(n - 1)]
E += [(r * n + c + 1, (r + 1) * n + c + 1, w()) for r in range(n - 1) for c in range(n)]
write("malla-250k", n * n, E, 1, n * n)

# 1M de nodos con aristas al azar: casi todo aislado, sin camino (explora todo)
write("disperso-1M", 1_000_000,
      [(random.randint(1, 1_000_000), random.randint(1, 1_000_000), w()) for _ in range(500_000)],
      1, 1_000_000)

# pocos nodos, muchas aristas: estresa la cola (3 tamanos para ver como escala)
for e in (100_000, 300_000, 500_000):
    write(f"denso-50k-E{e//1000}k", 50_000,
          [(random.randint(1, 50_000), random.randint(1, 50_000), w()) for _ in range(e)],
          1, 50_000)

In [ ]:
%%writefile chart.py
"""resultados.csv -> comparacion.html (SVG inline, sin dependencias).
Uso: python3 chart.py resultados.csv comparacion.html"""
import csv, sys, html
from collections import OrderedDict

src = sys.argv[1] if len(sys.argv) > 1 else "resultados.csv"
dst = sys.argv[2] if len(sys.argv) > 2 else "comparacion.html"

COL = {"parseo": "var(--neutral)", "Dial+mascara": "var(--series-1)",
       "Dial+barrido": "var(--series-2)", "Heap binario": "var(--series-3)",
       "Heap 4-ario": "var(--series-4)"}

rows = list(csv.DictReader(open(src)))
facets = OrderedDict()
for r in rows:
    facets.setdefault((r["grafo"], r["V"], r["E"]), []).append((r["variante"], float(r["ms_med"]), r["dist"]))

BH, GAP, LBL, PAD = 20, 8, 118, 54   # alto de barra, separacion, columna de etiquetas, margen derecho

def bar_path(x, y, w, h, r=4):
    if w < r: return f'<rect x="{x}" y="{y}" width="{max(w,0.8):.1f}" height="{h}"/>'
    return (f'<path d="M{x},{y}h{w-r:.1f}a{r},{r} 0 0 1 {r},{r}v{h-2*r}'
            f'a{r},{r} 0 0 1 {-r},{r}H{x}z"/>')

def facet(key, data):
    name, V, E = key
    hi = max(v for _, v, _ in data) or 1
    W, H = 440, len(data) * (BH + GAP) + 6
    plot = W - LBL - PAD
    out = [f'<figure class="facet"><figcaption><b>{html.escape(name[:-4])}</b>'
           f'<span>V={int(V):,} · E={int(E):,} · dist={data[0][2] if data[0][2]!="-" else data[1][2]}</span></figcaption>'
           f'<svg viewBox="0 0 {W} {H}" role="img">']
    for i, (var, ms, _) in enumerate(data):
        y = i * (BH + GAP) + 3
        w = plot * ms / hi
        out.append(f'<text class="lbl" x="{LBL-8}" y="{y+BH*0.72}" text-anchor="end">{html.escape(var)}</text>')
        out.append(f'<g fill="{COL.get(var,"var(--series-1)")}">{bar_path(LBL, y, w, BH)}</g>')
        out.append(f'<text class="val" x="{LBL+w+6:.1f}" y="{y+BH*0.72}">{ms:.2f} ms</text>')
    out.append("</svg></figure>")
    return "".join(out)

table = ["<table><thead><tr>" + "".join(f"<th>{h}</th>" for h in rows[0]) + "</tr></thead><tbody>"]
for r in rows:
    table.append("<tr>" + "".join(f"<td>{html.escape(v)}</td>" for v in r.values()) + "</tr>")
table.append("</tbody></table>")

open(dst, "w").write(f"""<!doctype html><html lang="es"><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Dijkstra: comparacion de colas</title>
<style>
:root{{color-scheme:light;--surface-1:#fcfcfb;--surface-2:#f4f3f0;--text-primary:#0b0b0b;
--text-secondary:#52514e;--border:#dedcd5;--neutral:#8a8880;
--series-1:#2a78d6;--series-2:#eb6834;--series-3:#1baf7a;--series-4:#eda100}}
@media (prefers-color-scheme:dark){{:root:not([data-theme="light"]){{color-scheme:dark;
--surface-1:#1a1a19;--surface-2:#232321;--text-primary:#f0efec;--text-secondary:#c3c2b7;
--border:#3a3a37;--neutral:#8a8880;--series-1:#3987e5;--series-2:#d95926;--series-3:#199e70;--series-4:#c98500}}}}
body{{margin:0;padding:24px 16px;background:var(--surface-1);color:var(--text-primary);
font:15px/1.5 ui-sans-serif,system-ui,sans-serif}}
main{{max-width:1000px;margin:0 auto}}
h1{{font-size:1.4rem;margin:0 0 .25rem}}
p.sub{{color:var(--text-secondary);margin:0 0 1.5rem}}
.grid{{display:grid;gap:20px;grid-template-columns:repeat(auto-fit,minmax(330px,1fr))}}
.facet{{margin:0;background:var(--surface-2);border:1px solid var(--border);border-radius:10px;padding:12px 14px}}
figcaption{{display:flex;flex-direction:column;margin-bottom:8px}}
figcaption span{{color:var(--text-secondary);font-size:.78rem}}
svg{{width:100%;height:auto;overflow:visible}}
.lbl{{fill:var(--text-secondary);font-size:11px}}
.val{{fill:var(--text-primary);font-size:11px;font-weight:600;font-variant-numeric:tabular-nums}}
table{{border-collapse:collapse;font-size:.82rem;margin-top:10px;width:100%}}
th,td{{text-align:left;padding:4px 10px;border-bottom:1px solid var(--border)}}
details{{margin-top:28px}}
</style>
<main>
<h1>Dijkstra: parseo vs. cola de prioridad</h1>
<p class="sub">Milisegundos (mediana). Cada barra esta etiquetada con su valor; la escala es
propia de cada grafo. <b>parseo</b> = leer el archivo + construir las listas de adyacencia,
una sola vez por ejecucion.</p>
<div class="grid">{"".join(facet(k, v) for k, v in facets.items())}</div>
<details><summary>Ver los datos (min y mediana)</summary>{"".join(table)}</details>
</main></html>""")
print(f"-> {dst}")

In [ ]:
%%writefile test.sh
#!/bin/sh
# Compila y verifica: casos a mano + comparacion contra una referencia en Python.
set -e
cd "$(dirname "$0")"
gcc -O2 -march=native -o dijkstra dijkstra.c

chk() { # chk <esperado> <archivo>
  got=$(./dijkstra "$2"); [ "$got" = "$1" ] && echo "ok  $2 -> $got" \
    || { echo "FALLO $2: esperado '$1', obtenido '$got'"; exit 1; }
}

mkdir -p t
printf '4 4\n1 4\na\nb\nc\nd\n1 2 3\n2 4 5\n1 3 2\n3 4 9\n' > t/basico.txt   # 1-2-4 = 8
printf '4 1\n1 4\n1\n2\n3\n4\n1 2 7\n'                       > t/sincamino.txt
printf '3 1\n2 2\n1\n2\n3\n1 2 4\n'                          > t/mismo.txt
printf '2 1\n1 2\n1\n2\n1 2 100\n'                           > t/unaarista.txt
chk 8               t/basico.txt
chk "No hay camino" t/sincamino.txt
chk 0               t/mismo.txt
chk 100             t/unaarista.txt
echo 0 | ./dijkstra - < t/basico.txt >/dev/null && echo "ok  stdin"

# las cuatro colas deben dar lo mismo
for q in dial barrido heap heap4; do
  got=$(./dijkstra t/basico.txt $q)
  [ "$got" = "8" ] || { echo "FALLO cola $q: obtenido '$got'"; exit 1; }
done
echo "ok  dial / barrido / heap / heap4 coinciden"
./dijkstra t/basico.txt lista 2>/dev/null && { echo "FALLO: cola invalida no dio error"; exit 1; }
echo "ok  cola invalida -> error"

# TIEMPO=1 no debe contaminar stdout
out=$(TIEMPO=1 ./dijkstra t/basico.txt 2>/dev/null)
err=$(TIEMPO=1 ./dijkstra t/basico.txt 2>&1 >/dev/null)
[ "$out" = "8" ] && [ -n "$err" ] || { echo "FALLO TIEMPO=1: stdout='$out' stderr='$err'"; exit 1; }
echo "ok  TIEMPO=1 -> stdout limpio, tiempos en stderr"

# aleatorios contra referencia
python3 - <<'PY'
import random, subprocess, heapq
for it in range(40):
    V=random.randint(2,60); E=random.randint(1,120)
    es=[(random.randint(1,V),random.randint(1,V),random.randint(1,100)) for _ in range(E)]
    s,t=random.randint(1,V),random.randint(1,V)
    txt="%d %d\n%d %d\n"%(V,E,s,t)+"".join("n%d\n"%i for i in range(1,V+1))+"".join("%d %d %d\n"%e for e in es)
    g={i:[] for i in range(1,V+1)}
    for u,v,w in es: g[u].append((v,w)); g[v].append((u,w))
    d={s:0}; q=[(0,s)]
    while q:
        dd,u=heapq.heappop(q)
        if dd>d.get(u,1<<60): continue
        for v,w in g[u]:
            if dd+w<d.get(v,1<<60): d[v]=dd+w; heapq.heappush(q,(dd+w,v))
    exp=str(d[t]) if t in d else "No hay camino"
    got=subprocess.run(["./dijkstra"],input=txt,capture_output=True,text=True).stdout.strip()
    assert got==exp, (it,exp,got,txt)
print("ok  40 grafos aleatorios vs referencia")
PY

# grande y determinista: cadena de 500.000 aristas de peso 100 (distancia maxima 5e7)
# ademas ejercita el salto por bloques de 1.000.000 de etiquetas
python3 -c "
V,E=1000000,500000
L=['%d %d'%(V,E),'1 %d'%(E+1)]+['n%d'%i for i in range(1,V+1)]
L+=['%d %d 100'%(i,i+1) for i in range(1,E+1)]
open('t/grande.txt','w').write('\\n'.join(L)+'\\n')"
chk 50000000 t/grande.txt
echo "--- tiempo total con V=1.000.000 / E=500.000 (15 MB):"
time ./dijkstra t/grande.txt

In [ ]:
%%writefile bench.sh
#!/bin/sh
# Mide parseo y las 4 colas sobre todos los grafos de t/ -> resultados.csv + comparacion.html
# Uso: ./bench.sh [repeticiones]   (por defecto 7; se reporta la mediana)
set -e
cd "$(dirname "$0")"
R=${1:-7}
gcc -O2 -march=native -o bench bench.c
[ -f t/cadena-1M.txt ] || python3 gen.py

echo "grafo,V,E,variante,ms_min,ms_med,dist" > resultados.csv
for f in t/cadena-1M.txt t/malla-250k.txt t/disperso-1M.txt \
         t/denso-50k-E100k.txt t/denso-50k-E300k.txt t/denso-50k-E500k.txt; do
    printf '%s... ' "$f" >&2
    ./bench "$f" "$R" >> resultados.csv
    echo ok >&2
done

python3 chart.py resultados.csv comparacion.html
column -s, -t resultados.csv 2>/dev/null || cat resultados.csv   # column puede no existir
echo "-> resultados.csv y comparacion.html (abrir en el navegador)"

## 2. Entorno y compilación

Se muestra en qué máquina de Colab se midió, porque los tiempos dependen de ella.

In [ ]:
!chmod +x test.sh bench.sh
!gcc -O2 -march=native -o dijkstra dijkstra.c && echo "compilado OK"
!lscpu | grep -E "^Model name|^CPU\(s\):" ; free -h | head -2

## 3. El ejemplo del enunciado

`V=10`, `E=2`, camino de `3` a `7`: las aristas `3-5` (14) y `5-7` (7) dan **21**.

In [ ]:
%%writefile ejemplo.txt
10 2
3 7
1
2
3
4
5
6
7
8
9
10
3 5 14
5 7 7

In [ ]:
!./dijkstra ejemplo.txt

## 4. Correctitud

`test.sh` comprueba:

- los casos borde: `s == t`, grafo sin camino, una sola arista, lectura por `stdin`;
- que las **cuatro** colas de prioridad implementadas den el mismo resultado;
- **40 grafos aleatorios** contra un Dijkstra de referencia escrito en Python (`heapq`);
- un grafo de `V = 1.000.000` con todos los pesos en 100, cuya distancia es exactamente
  5×10⁷ (verifica que no haya desbordamiento en 32 bits).

In [ ]:
!./test.sh

## 5. Tiempo con la entrada más grande del enunciado

`V = 1.000.000`, `E = 500.000`: un archivo de ~15 MB con un millón de etiquetas.
`TIEMPO=1` hace que el programa imprima en **stderr** el desglose de tiempos
(cuesta 3 llamadas a `clock_gettime`, unos 93 ns, es decir 0,0001 ms);
por **stdout** sale únicamente la distancia.

In [ ]:
# cadena de 500.000 aristas de peso 100 -> distancia 50.000.000
import time
V, E = 1_000_000, 500_000
t0 = time.time()
with open("grande.txt", "w") as f:
    f.write(f"{V} {E}\n1 {E+1}\n")
    f.write("".join(f"n{i}\n" for i in range(1, V + 1)))
    f.write("".join(f"{i} {i+1} 100\n" for i in range(1, E + 1)))
print(f"archivo generado en {time.time()-t0:.1f} s")
!ls -lh grande.txt

In [ ]:
# 5 corridas: la primera suele salir mas lenta (cache frio y CPU compartida de Colab)
!for i in 1 2 3 4 5; do TIEMPO=1 ./dijkstra grande.txt; done

### Lectura de esos números

`parseo + algoritmo = total`. El parseo recorre los ~15 millones de bytes del archivo uno por uno;
Dijkstra solo recorre la estructura ya construida. Por eso pesan parecido: **el cuello de botella
real es la entrada, no el algoritmo**.

El `total` no incluye el arranque ni el cierre del proceso (~4 ms), así que es menor que lo que
mediría `time` desde fuera.

## 6. ¿Por qué Dial y no un heap?

`queues.h` tiene **cuatro** colas de prioridad intercambiables sobre el mismo Dijkstra, y
`bench.c` las cronometra sobre el mismo grafo ya parseado (el parseo se mide aparte):

| cola | qué es |
|---|---|
| `dial` | 101 cubetas circulares + máscara de bits para saltar las vacías |
| `barrido` | las mismas cubetas, pero buscando la siguiente recorriéndolas una por una |
| `heap` | heap binario, clave `(dist<<20)\|nodo` empaquetada en un `uint64` |
| `heap4` | heap 4-ario (menos niveles y hijos contiguos: menos fallos de caché) |

Se prueban seis formas de grafo (cadena larga, malla 500×500, disperso de 1M nodos y denso en
tres tamaños). Tarda uno o dos minutos: genera ~70 MB de grafos de prueba.

In [ ]:
!./bench.sh 5

In [ ]:
import pandas as pd
df = pd.read_csv("resultados.csv")
tabla = df.pivot(index="grafo", columns="variante", values="ms_med")
tabla[["parseo", "Dial+mascara", "Dial+barrido", "Heap binario", "Heap 4-ario"]]

In [ ]:
from IPython.display import HTML
HTML(open("comparacion.html").read())   # el mismo cuadro como grafico

### Conclusiones del benchmark

- **La máscara de bits solo importa en la cadena** (28,9 → 8,0 ms): ahí cada arista pesa ~50, así que
  entre dos extracciones hay que saltar ~50 cubetas vacías; 500.000 × 50 = 25 millones de pasos en
  vacío. En la malla y en el denso la frontera tiene miles de nodos con distancias contiguas y la
  cubeta siguiente está casi siempre a un paso.
- **El heap gana en la cadena** (2,7 vs 8,0 ms) porque ahí la cola tiene 1 o 2 elementos: `log n ≈ 1`.
  **Dial gana 3x en la malla** (7,1 vs 23,2 ms), donde la frontera tiene ~1.000 nodos y cada operación
  del heap salta a `i/2, i/4, …` provocando un fallo de caché por nivel.
- **El heap 4-ario le gana al binario** (15,0 vs 23,2 ms) haciendo *más* comparaciones: tiene la mitad
  de niveles y los 4 hijos son contiguos. En hardware moderno gana el que toca menos memoria.
- Se entrega **Dial** por criterio *minimax*: su peor caso medido es 8 ms contra los 23 ms del heap.
  No se sabe qué grafo usará el evaluador, así que se minimiza el peor caso.
- En el grafo disperso Dijkstra tarda **0,00 ms** y el archivo 10 ms: el peor caso del algoritmo y
  el peor caso del programa no son el mismo grafo.

*(Los valores citados son los medidos en un i9-13900H; en Colab serán más altos, pero las
proporciones entre colas se mantienen.)*

## 7. Probar con otro archivo

Sube un archivo con el formato del enunciado y se ejecuta sobre él.
También se puede elegir la cola: `./dijkstra archivo.txt heap`.

In [ ]:
from google.colab import files
subido = files.upload()
nombre = next(iter(subido))
!TIEMPO=1 ./dijkstra "$nombre"